In [5]:
import numpy as np
import faiss
from scipy.sparse import csr_matrix, vstack, hstack
import networkx as nx
from tqdm import tqdm
import time
from typing import List, Tuple, Dict, Set, Optional, Union
import matplotlib.pyplot as plt


class LowDimNNGraph:
    """
    A class to compute and store nearest neighbor graphs in low-dimensional spaces using FAISS.
    """
    
    def __init__(self, dim_idx: int, points: np.ndarray, n_neighbors: int = 10, 
                 index_type: str = "hnsw", metric: str = "l2"):
        """
        Initialize a low-dimensional nearest neighbor graph.
        
        Args:
            dim_idx: Index identifier for this low-dimensional set
            points: Low-dimensional points of shape (N, d)
            n_neighbors: Number of nearest neighbors to find for each point
            index_type: FAISS index type to use ("hnsw", "ivf", or "flat")
            metric: Distance metric to use ("l2" or "ip" for inner product)
        """
        self.dim_idx = dim_idx
        self.points = points.astype(np.float32)  # FAISS requires float32
        self.n_points = points.shape[0]
        self.dim = points.shape[1]
        self.n_neighbors = n_neighbors
        self.index_type = index_type
        self.metric = metric
        
        # Create and build the FAISS index
        self.index = self._build_faiss_index()
        self.index.add(self.points)
        
        # Compute the nearest neighbor graph
        self.nn_distances, self.nn_indices = self._compute_nn_graph()
        
        # Create an adjacency matrix representation
        self.adjacency_matrix = self._create_adjacency_matrix()
        
    def _build_faiss_index(self) -> faiss.Index:
        """Build an appropriate FAISS index based on index_type."""
        if self.metric == "l2":
            if self.index_type == "flat":
                return faiss.IndexFlatL2(self.dim)
            elif self.index_type == "hnsw":
                index = faiss.IndexHNSWFlat(self.dim, 32)  # 32 is M (max connections per node)
                index.hnsw.efConstruction = 40  # More accurate construction
                index.hnsw.efSearch = 16  # More accurate search
                return index
            elif self.index_type == "ivf":
                # For IVF, we need a quantizer and to train the index
                quantizer = faiss.IndexFlatL2(self.dim)
                nlist = min(int(np.sqrt(self.n_points)), 100)  # Number of clusters
                index = faiss.IndexIVFFlat(quantizer, self.dim, nlist, faiss.METRIC_L2)
                # Need to train IVF indices
                if self.n_points >= 2 * nlist:  # Need enough data to train
                    index.train(self.points)
                return index
            else:
                raise ValueError(f"Unsupported index type: {self.index_type}")
        elif self.metric == "ip":  # Inner product
            if self.index_type == "flat":
                return faiss.IndexFlatIP(self.dim)
            # Add other index types for IP if needed
            else:
                raise ValueError(f"Unsupported index type for IP metric: {self.index_type}")
        else:
            raise ValueError(f"Unsupported metric: {self.metric}")
    
    def _compute_nn_graph(self) -> Tuple[np.ndarray, np.ndarray]:
        """Compute the k-nearest neighbor graph using FAISS."""
        # For each point, find its n_neighbors nearest neighbors (including itself)
        k = min(self.n_neighbors + 1, self.n_points)  # +1 because the point itself is included
        distances, indices = self.index.search(self.points, k)
        
        # Remove self-connections (the first column, which is the point itself)
        return distances[:, 1:], indices[:, 1:]
    
    def _create_adjacency_matrix(self) -> csr_matrix:
        """Create a sparse adjacency matrix from NN indices."""
        rows = np.repeat(np.arange(self.n_points), self.nn_indices.shape[1])
        cols = self.nn_indices.flatten()
        data = np.ones_like(cols)  # Binary adjacency (1 for connected)
        
        # Create a sparse adjacency matrix
        adj_matrix = csr_matrix(
            (data, (rows, cols)), 
            shape=(self.n_points, self.n_points)
        )
        
        return adj_matrix
    
    def get_neighbors(self, point_idx: int) -> np.ndarray:
        """Get the indices of neighbors for a given point index."""
        return self.nn_indices[point_idx]
    
    def get_point(self, point_idx: int) -> np.ndarray:
        """Get the coordinates of a point by its index."""
        return self.points[point_idx]
    
    def query(self, query_points: np.ndarray, k: Optional[int] = None) -> Tuple[np.ndarray, np.ndarray]:
        """Query the FAISS index for k nearest neighbors of query points."""
        if k is None:
            k = self.n_neighbors
        
        k = min(k, self.n_points)
        distances, indices = self.index.search(query_points.astype(np.float32), k)
        return distances, indices


class CartesianNNGraph:
    """
    High-dimensional nearest neighbor graph constructed via Cartesian product
    of multiple low-dimensional nearest neighbor graphs.
    """
    
    def __init__(self, low_dim_graphs: List[LowDimNNGraph]):
        """
        Initialize the high-dimensional graph using multiple low-dimensional NN graphs.
        
        Args:
            low_dim_graphs: List of LowDimNNGraph objects
        """
        self.low_dim_graphs = low_dim_graphs
        self.n_low_dims = len(low_dim_graphs)
        
        # Verify that all low-dimensional graphs have the same number of points
        n_points_set = {graph.n_points for graph in low_dim_graphs}
        if len(n_points_set) != 1:
            raise ValueError("All low-dimensional graphs must have the same number of points")
        self.n_points = next(iter(n_points_set))
        
        # Total dimensionality of the high-dimensional space
        self.total_dim = sum(graph.dim for graph in low_dim_graphs)
        
        # Create a mapping function for multi-dimensional indices
        # This is needed for efficient queries in the high-dimensional space
        self._build_high_dim_graph()
    
    def _build_high_dim_graph(self):
        """Build the high-dimensional graph structure."""
        # Store the full adjacency matrix of the high-dimensional graph
        # We'll construct this from the low-dimensional adjacency matrices
        
        # Initialize an empty adjacency matrix
        self.hd_adjacency_matrix = None
        
        # For each low-dimensional graph, create connections in the high-dimensional space
        # based on the low-dimensional nearest neighbor relationships
        for i, low_dim_graph in enumerate(self.low_dim_graphs):
            print(f"Processing low-dimensional graph {i+1}/{self.n_low_dims}...")
            
            # Get the adjacency matrix of the current low-dimensional graph
            ld_adjacency = low_dim_graph.adjacency_matrix
            
            if self.hd_adjacency_matrix is None:
                # Initialize with the first graph's adjacency
                self.hd_adjacency_matrix = ld_adjacency.copy()
            else:
                # Merge with existing connections via logical OR operation
                # This combines all the low-dimensional edge connections
                self.hd_adjacency_matrix = self.hd_adjacency_matrix.maximum(ld_adjacency)
        
        # Convert to NetworkX graph for easier traversal
        print("Converting to NetworkX graph...")
        self.graph = nx.from_scipy_sparse_array(self.hd_adjacency_matrix, create_using=nx.Graph)
        
        print(f"High-dimensional graph constructed with {self.n_points} nodes and "
              f"{self.graph.number_of_edges()} edges")
    
    def get_high_dim_point(self, point_idx: int) -> np.ndarray:
        """
        Get the high-dimensional coordinates for a point by concatenating 
        its coordinates from all low-dimensional spaces.
        """
        parts = [graph.get_point(point_idx) for graph in self.low_dim_graphs]
        return np.concatenate(parts)
    
    def get_neighbors(self, point_idx: int) -> List[int]:
        """Get all neighbors of a point in the high-dimensional graph."""
        return list(self.graph.neighbors(point_idx))
    
    def query(self, query_point: np.ndarray, k: int = 10) -> Tuple[np.ndarray, np.ndarray]:
        """
        Find k-nearest neighbors of a high-dimensional query point by:
        1. Splitting the query into low-dimensional components
        2. Querying each low-dimensional FAISS index
        3. Combining and refining results
        
        Args:
            query_point: High-dimensional query vector of shape (d_total,)
            k: Number of nearest neighbors to return
            
        Returns:
            Tuple of (distances, indices) arrays
        """
        # Split the query point into low-dimensional components
        dim_offset = 0
        low_dim_queries = []
        for graph in self.low_dim_graphs:
            low_dim_query = query_point[dim_offset:dim_offset + graph.dim]
            low_dim_queries.append(low_dim_query.reshape(1, -1))
            dim_offset += graph.dim
        
        # Query each low-dimensional graph
        # This gives us candidate nearest neighbors in each low-dimensional space
        candidate_sets = []
        for i, (graph, query) in enumerate(zip(self.low_dim_graphs, low_dim_queries)):
            _, indices = graph.query(query, k=k)
            candidate_sets.append(set(indices[0]))
        
        # Combine candidate sets
        # We take the union of all candidate points from all low-dimensional spaces
        all_candidates = set().union(*candidate_sets)
        
        # Get high-dimensional coordinates for all candidates
        candidate_points = np.array([self.get_high_dim_point(idx) for idx in all_candidates])
        candidate_indices = np.array(list(all_candidates))
        
        # Calculate exact distances in the high-dimensional space
        distances = np.linalg.norm(candidate_points - query_point, axis=1)
        
        # Sort by distance and return top k
        sorted_idx = np.argsort(distances)[:k]
        top_distances = distances[sorted_idx]
        top_indices = candidate_indices[sorted_idx]
        
        return top_distances, top_indices
    
    def refine_search_by_graph_traversal(self, query_point: np.ndarray, 
                                         initial_neighbors: np.ndarray, 
                                         max_hops: int = 5, 
                                         max_candidates: int = 100,
                                         k: int = 10) -> Tuple[np.ndarray, np.ndarray]:
        """
        Refine search results using graph traversal from initial neighbors.
        
        Args:
            query_point: High-dimensional query vector
            initial_neighbors: Initial set of neighbor indices
            max_hops: Maximum number of hops in graph traversal
            max_candidates: Maximum number of candidates to consider
            k: Number of nearest neighbors to return
            
        Returns:
            Tuple of (distances, indices) arrays
        """
        # Initialize set of candidates with initial neighbors
        candidates = set(initial_neighbors)
        visited = set(initial_neighbors)
        
        # Perform graph traversal up to max_hops
        current_frontier = set(initial_neighbors)
        for _ in range(max_hops):
            if len(candidates) >= max_candidates:
                break
                
            next_frontier = set()
            for node in current_frontier:
                # Get neighbors of the current node
                neighbors = self.get_neighbors(node)
                for neighbor in neighbors:
                    if neighbor not in visited:
                        next_frontier.add(neighbor)
                        candidates.add(neighbor)
                        visited.add(neighbor)
                        
                        if len(candidates) >= max_candidates:
                            break
                if len(candidates) >= max_candidates:
                    break
                    
            current_frontier = next_frontier
            if len(current_frontier) == 0:
                break
        
        # Calculate distances to all candidates
        candidate_list = list(candidates)
        candidate_points = np.array([self.get_high_dim_point(idx) for idx in candidate_list])
        distances = np.linalg.norm(candidate_points - query_point, axis=1)
        
        # Sort by distance and return top k
        sorted_idx = np.argsort(distances)[:k]
        top_distances = distances[sorted_idx]
        top_indices = np.array(candidate_list)[sorted_idx]
        
        return top_distances, top_indices


def generate_synthetic_data(n_points: int = 1000, 
                            n_dims: int = 3, 
                            dim_sizes: Optional[List[int]] = None,
                            random_seed: int = 42) -> List[np.ndarray]:
    """
    Generate synthetic data for testing the algorithm.
    
    Args:
        n_points: Number of points in each dimension
        n_dims: Number of independent low-dimensional spaces
        dim_sizes: List of dimensionality for each low-dimensional space
        random_seed: Random seed for reproducibility
        
    Returns:
        List of numpy arrays, each representing a low-dimensional point set
    """
    np.random.seed(random_seed)
    
    if dim_sizes is None:
        # Default: create dimensions of size 2, 3, 4, etc.
        dim_sizes = [i + 2 for i in range(n_dims)]
    
    data_sets = []
    for dim in dim_sizes:
        # Generate random points in the given dimension
        points = np.random.rand(n_points, dim)
        data_sets.append(points)
    
    return data_sets


def benchmark_methods(data_sets: List[np.ndarray], 
                      n_neighbors: int = 10,
                      n_test_queries: int = 100,
                      random_seed: int = 42) -> Dict:
    """
    Benchmark the Cartesian NN graph method against brute force.
    
    Args:
        data_sets: List of low-dimensional datasets
        n_neighbors: Number of neighbors to find
        n_test_queries: Number of test queries to perform
        random_seed: Random seed for reproducibility
        
    Returns:
        Dictionary with benchmark results
    """
    np.random.seed(random_seed)
    n_points = data_sets[0].shape[0]
    total_dim = sum(data.shape[1] for data in data_sets)
    
    # Construct the full high-dimensional dataset
    print("Constructing high-dimensional dataset...")
    high_dim_data = np.concatenate([data for data in data_sets], axis=1)
    
    print("Building low-dimensional NN graphs...")
    # Create low-dimensional NN graphs
    low_dim_graphs = []
    for i, data in enumerate(data_sets):
        print(f"Building graph {i+1}/{len(data_sets)}...")
        graph = LowDimNNGraph(i, data, n_neighbors=n_neighbors)
        low_dim_graphs.append(graph)
    
    print("Building Cartesian NN graph...")
    # Construct the Cartesian NN graph
    cart_graph = CartesianNNGraph(low_dim_graphs)
    
    # Generate random test queries
    test_idx = np.random.choice(n_points, n_test_queries, replace=False)
    test_queries = high_dim_data[test_idx]
    
    # Benchmark: Cartesian NN graph method
    print("Benchmarking Cartesian NN graph method...")
    cart_times = []
    for i, query in enumerate(test_queries):
        start_time = time.time()
        _, initial_neighbors = cart_graph.query(query, k=n_neighbors)
        _, _ = cart_graph.refine_search_by_graph_traversal(
            query, initial_neighbors, max_hops=2, k=n_neighbors)
        end_time = time.time()
        cart_times.append(end_time - start_time)
        if (i + 1) % 10 == 0:
            print(f"Processed {i+1}/{n_test_queries} queries")
    
    # Benchmark: Brute force method
    print("Benchmarking brute force method...")
    brute_times = []
    for i, query in enumerate(test_queries):
        start_time = time.time()
        distances = np.linalg.norm(high_dim_data - query, axis=1)
        _ = np.argsort(distances)[:n_neighbors]
        end_time = time.time()
        brute_times.append(end_time - start_time)
        if (i + 1) % 10 == 0:
            print(f"Processed {i+1}/{n_test_queries} queries")
    
    # Compute accuracy (overlap between Cartesian and brute force results)
    print("Computing accuracy...")
    accuracy = []
    for i, query in enumerate(test_queries):
        # Cartesian method
        _, cart_indices = cart_graph.query(query, k=n_neighbors)
        _, refined_indices = cart_graph.refine_search_by_graph_traversal(
            query, cart_indices, max_hops=2, k=n_neighbors)
        cart_set = set(refined_indices)
        
        # Brute force
        distances = np.linalg.norm(high_dim_data - query, axis=1)
        brute_indices = np.argsort(distances)[:n_neighbors]
        brute_set = set(brute_indices)
        
        # Compute overlap
        overlap = len(cart_set.intersection(brute_set)) / n_neighbors
        accuracy.append(overlap)
    
    results = {
        "cartesian_times": cart_times,
        "brute_times": brute_times,
        "accuracy": accuracy,
        "avg_cartesian_time": np.mean(cart_times),
        "avg_brute_time": np.mean(brute_times),
        "speedup": np.mean(brute_times) / np.mean(cart_times),
        "avg_accuracy": np.mean(accuracy)
    }
    
    return results


def visualize_benchmark(results: Dict):
    """Visualize benchmark results."""
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    
    # Plot query times
    axs[0].boxplot([results["cartesian_times"], results["brute_times"]])
    axs[0].set_yscale('log')
    axs[0].set_ylabel('Query Time (seconds, log scale)')
    axs[0].set_xticklabels(['Cartesian', 'Brute Force'])
    axs[0].set_title('Query Time Comparison')
    
    # Plot average times
    methods = ['Cartesian', 'Brute Force']
    times = [results["avg_cartesian_time"], results["avg_brute_time"]]
    axs[1].bar(methods, times)
    axs[1].set_ylabel('Avg. Query Time (seconds)')
    axs[1].set_title(f'Average Query Time\nSpeedup: {results["speedup"]:.2f}x')
    
    # Plot accuracy histogram
    axs[2].hist(results["accuracy"], bins=10, range=(0, 1))
    axs[2].set_xlabel('Accuracy (overlap with brute force)')
    axs[2].set_ylabel('Count')
    axs[2].set_title(f'Accuracy Distribution\nAvg: {results["avg_accuracy"]:.2f}')
    
    plt.tight_layout()
    plt.savefig('benchmark_results.png')
    plt.close()


def main():
    # Generate synthetic data
    print("Generating synthetic data...")
    n_points = 1000000
    dim_sizes = [3, 4, 5, 16]  # Three low-dimensional spaces
    data_sets = generate_synthetic_data(n_points=n_points, dim_sizes=dim_sizes)
    
    # Benchmark the methods
    print("Benchmarking methods...")
    results = benchmark_methods(
        data_sets=data_sets,
        n_neighbors=10,
        n_test_queries=50
    )
    
    # Print results
    print("\nBenchmark Results:")
    print(f"Average Cartesian query time: {results['avg_cartesian_time']:.6f} seconds")
    print(f"Average Brute Force query time: {results['avg_brute_time']:.6f} seconds")
    print(f"Speedup: {results['speedup']:.2f}x")
    print(f"Average accuracy: {results['avg_accuracy']:.2f}")
    
    # Visualize results
    print("Visualizing results...")
    visualize_benchmark(results)
    
    print("Done!")


if __name__ == "__main__":
    main()

Generating synthetic data...
Benchmarking methods...
Constructing high-dimensional dataset...
Building low-dimensional NN graphs...
Building graph 1/4...
Building graph 2/4...
Building graph 3/4...
Building graph 4/4...
Building Cartesian NN graph...
Processing low-dimensional graph 1/4...
Processing low-dimensional graph 2/4...
Processing low-dimensional graph 3/4...
Processing low-dimensional graph 4/4...
Converting to NetworkX graph...
High-dimensional graph constructed with 1000000 nodes and 24129166 edges
Benchmarking Cartesian NN graph method...
Processed 10/50 queries
Processed 20/50 queries
Processed 30/50 queries
Processed 40/50 queries
Processed 50/50 queries
Benchmarking brute force method...
Processed 10/50 queries
Processed 20/50 queries
Processed 30/50 queries
Processed 40/50 queries
Processed 50/50 queries
Computing accuracy...

Benchmark Results:
Average Cartesian query time: 0.000509 seconds
Average Brute Force query time: 0.170353 seconds
Speedup: 334.50x
Average accu